# R4 · Logique, ensembles, Σ et Π — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r4_logique_ensembles/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r4_logique_ensembles/03_notebook.ipynb`, créée par `python tools/start_chapter.py R4`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R4.1 à R4.8, R4.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R4.17 | Tables de vérité et implication · `truth_table`, `implies` | 🔨 | ★ | 30 |
|  | R4.19 | Chercher un contre-exemple de 0 à N · `find_counterexample` | 🔨 | ★★ | 30 |
|  | R4.21 | Prédire : n² + n + 41 est-il toujours un nombre premier ? | 🔮 | ★ | 15 |
| B | R4.20 | Les ensembles de Python : `set`, `\|`, `&`, `-` | 📦 | ★ | 10 |
|  | R4.23 | Diagrammes de Venn | 🎨 | ★ | 15 |
| C | R4.18 | Σ et Π en Python · `sum_over`, `prod_over` | 🔨 | ★ | 25 |
|  | R4.22 | Vérifier une formule de somme de 1 à 1 000, puis la démontrer | 🔬 | ★★ | 20 |
|  | R4.24 | Chasse à l'erreur : une « démonstration » par récurrence qui prouve le faux | 🐛 | ★★ | 20 |
|  | R4.25 | Rédiger une démonstration propre en LaTeX | 🛠️ | ★ | 20 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R4")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : tables de vérité, implications, quantificateurs, ensembles, sommes Σ et produits Π, contre-exemples, une loss moyenne.
- Écrire les cinq fonctions de `mymath/logic.py` et les tester.
- Voir une propriété tenir 40 fois puis tomber, vérifier une formule mille fois puis la démontrer, chasser l'erreur dans deux récurrences.

**Rappel express.** $P \Rightarrow Q$ n'est fausse que si P est vraie et Q fausse ; elle équivaut à sa contraposée $\text{non } Q \Rightarrow \text{non } P$, pas à sa réciproque. Nier : « pour tout » devient « il existe », « et » devient « ou ». $\lvert A \cup B \rvert = \lvert A \rvert + \lvert B \rvert - \lvert A \cap B \rvert$. $\sum_{k=m}^{n}$ a $n - m + 1$ termes, bornes incluses : `range(m, n + 1)`. Récurrence : initialisation, hérédité, conclusion.

## Partie 0 · Vérifier tes exercices papier (✏️ R4.1 à R4.8, 🔗 R4.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- une colonne de table de vérité par la liste de ses valeurs, dans l'ordre des lignes VV, VF, FV, FF : `[True, False, False, False]` pour « P et Q » (ou les lettres V et F, entre guillemets) ;
- vrai ou faux par `True` ou `False` ; un numéro de réponse proposée par un entier ;
- un ensemble par la liste de ses éléments, dans l'ordre que tu veux, la liste vide pour $\varnothing$ ;
- un nombre avec un point décimal, `-1.5` ; une fraction exacte entre guillemets, `"5/7"` ;
- une expression entre guillemets, `"3*n**2 + 1"` ; un intervalle entre guillemets, `"]-1 ; 4]"`.

Les réponses pas encore remplies affichent ⏳.

**Ex R4.1 — Tables de vérité de « et », « ou », « non » et de l'implication**

In [ ]:
# Ex R4.1: the values you found on paper
answer_R4_1a = ...  # a) a list of 4 values (rows VV, VF, FV, FF)
answer_R4_1b = ...  # b) a list of 4 values
answer_R4_1c = ...  # c) True or False
answer_R4_1d = ...  # d) a list of 4 values
answer_R4_1e = ...  # e) a list of 4 values
answer_R4_1f = ...  # f) a list of 4 values
answer_R4_1g = ...  # g) two letters, as text
answer_R4_1h = ...  # h) an integer
answer_R4_1i = ...  # i) an integer

for letter, answer in zip("abcdefghi", [answer_R4_1a, answer_R4_1b, answer_R4_1c, answer_R4_1d, answer_R4_1e, answer_R4_1f, answer_R4_1g, answer_R4_1h, answer_R4_1i]):
    wbm.check(f"R4.1{letter}", answer)

**Ex R4.2 — Réciproque, contraposée, négation : écrire les trois**

In [ ]:
# Ex R4.2: the values you found on paper
answer_R4_2a = ...  # a) the list [implication, converse, contrapositive, negation]
answer_R4_2b = ...  # b) a list of 4 values
answer_R4_2c = ...  # c) a list of 4 values
answer_R4_2d = ...  # d) a list of 4 values
answer_R4_2e = ...  # e) an integer
answer_R4_2f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_R4_2a, answer_R4_2b, answer_R4_2c, answer_R4_2d, answer_R4_2e, answer_R4_2f]):
    wbm.check(f"R4.2{letter}", answer)

**Ex R4.3 — Nier des phrases à deux quantificateurs**

In [ ]:
# Ex R4.3: the values you found on paper
answer_R4_3a = ...  # a) a number (1 to 4)
answer_R4_3b = ...  # b) True or False
answer_R4_3c = ...  # c) a number (1 to 4)
answer_R4_3d = ...  # d) True or False
answer_R4_3e = ...  # e) a list of 2 values
answer_R4_3f = ...  # f) a number (1 to 4)
answer_R4_3g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_R4_3a, answer_R4_3b, answer_R4_3c, answer_R4_3d, answer_R4_3e, answer_R4_3f, answer_R4_3g]):
    wbm.check(f"R4.3{letter}", answer)

**Ex R4.4 — Union, intersection, complémentaire et cardinal**

In [ ]:
# Ex R4.4: the values you found on paper
answer_R4_4a = ...  # a) a set, as a list (empty if it is empty)
answer_R4_4b = ...  # b) an integer
answer_R4_4c = ...  # c) a set, as a list
answer_R4_4d = ...  # d) an integer
answer_R4_4e = ...  # e) an integer
answer_R4_4f = ...  # f) a set, as a list
answer_R4_4g = ...  # g) an integer
answer_R4_4h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R4_4a, answer_R4_4b, answer_R4_4c, answer_R4_4d, answer_R4_4e, answer_R4_4f, answer_R4_4g, answer_R4_4h]):
    wbm.check(f"R4.4{letter}", answer)

**Ex R4.5 — Produit cartésien et couples**

In [ ]:
# Ex R4.5: the values you found on paper
answer_R4_5a = ...  # a) an integer
answer_R4_5b = ...  # b) an integer
answer_R4_5c = ...  # c) an integer
answer_R4_5d = ...  # d) an integer
answer_R4_5e = ...  # e) True or False
answer_R4_5f = ...  # f) an integer
answer_R4_5g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_R4_5a, answer_R4_5b, answer_R4_5c, answer_R4_5d, answer_R4_5e, answer_R4_5f, answer_R4_5g]):
    wbm.check(f"R4.5{letter}", answer)

**Ex R4.6 — Calculer des sommes Σ et des produits Π**

In [ ]:
# Ex R4.6: the values you found on paper
answer_R4_6a = ...  # a) an integer
answer_R4_6b = ...  # b) an integer
answer_R4_6c = ...  # c) an integer
answer_R4_6d = ...  # d) a fraction, as text
answer_R4_6e = ...  # e) an integer
answer_R4_6f = ...  # f) an integer
answer_R4_6g = ...  # g) an integer
answer_R4_6h = ...  # h) an integer
answer_R4_6i = ...  # i) an integer

for letter, answer in zip("abcdefghi", [answer_R4_6a, answer_R4_6b, answer_R4_6c, answer_R4_6d, answer_R4_6e, answer_R4_6f, answer_R4_6g, answer_R4_6h, answer_R4_6i]):
    wbm.check(f"R4.6{letter}", answer)

**Ex R4.7 — Développer, réindexer et découper une somme Σ ; calculer une somme double**

In [ ]:
# Ex R4.7: the values you found on paper
answer_R4_7a = ...  # a) an integer
answer_R4_7b = ...  # b) an integer
answer_R4_7c = ...  # c) an integer
answer_R4_7d = ...  # d) an expression in n, as text
answer_R4_7e = ...  # e) the list [p, q]
answer_R4_7f = ...  # f) an integer
answer_R4_7g = ...  # g) an integer
answer_R4_7h = ...  # h) a fraction, as text

for letter, answer in zip("abcdefgh", [answer_R4_7a, answer_R4_7b, answer_R4_7c, answer_R4_7d, answer_R4_7e, answer_R4_7f, answer_R4_7g, answer_R4_7h]):
    wbm.check(f"R4.7{letter}", answer)

**Ex R4.8 — Trouver un contre-exemple**

In [ ]:
# Ex R4.8: the values you found on paper
answer_R4_8a = ...  # a) an integer
answer_R4_8b = ...  # b) an integer
answer_R4_8c = ...  # c) an interval, as text
answer_R4_8d = ...  # d) an integer
answer_R4_8e = ...  # e) an integer
answer_R4_8f = ...  # f) a set, as a list

for letter, answer in zip("abcdef", [answer_R4_8a, answer_R4_8b, answer_R4_8c, answer_R4_8d, answer_R4_8e, answer_R4_8f]):
    wbm.check(f"R4.8{letter}", answer)

**Ex R4.14 — Écrire la loss moyenne d'un modèle avec Σ**

In [ ]:
# Ex R4.14: the values you found on paper
answer_R4_14a = ...  # a) an integer
answer_R4_14b = ...  # b) a number (or a fraction)
answer_R4_14c = ...  # c) a number (or a fraction)
answer_R4_14d = ...  # d) a number (or a fraction)
answer_R4_14e = ...  # e) a number (or a fraction)
answer_R4_14f = ...  # f) 1 or 2

for letter, answer in zip("abcdef", [answer_R4_14a, answer_R4_14b, answer_R4_14c, answer_R4_14d, answer_R4_14e, answer_R4_14f]):
    wbm.check(f"R4.14{letter}", answer)

## Partie A · La logique en Python

Fiche §1, §2 et §6. Tes premières fonctions de `mymath/logic.py` : une table de vérité, l'implication, la recherche d'un contre-exemple ; puis une propriété qui tient longtemps avant de tomber.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def is_prime(n):
    """True if the integer n is a prime number (it has exactly two divisors, 1 and itself)."""
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True

### Ex R4.17 — Tables de vérité et implication · `truth_table`, `implies` 🔨 ★ ⏱️ 30 min
**Objectif :** programmer une table de vérité pour n'importe quelle formule, et l'implication.  
**Prérequis :** fiche §1 et §2 ; workbookIA 0A §100.6.6 (`itertools`, aperçu) · **mymath :** `logic.py` · **Parcours :** C

> **Mode d'emploi `mymath`** (comme en R2 et R3). Ouvre ta copie `mon_travail/mymath/logic.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Les tests comparent tes résultats à SymPy et à une évaluation directe.

Dans `mon_travail/mymath/logic.py` :
- `truth_table(formula, n)` : une `ValueError` si `n < 1` ; sinon, pour chaque combinaison de `n` booléens, dans l'ordre de `itertools.product([True, False], repeat=n)` (toutes vraies d'abord), la ligne `(values, result)`, où `values` est le tuple des `n` booléens et `result` la valeur de la formule. Pour appeler la formule avec les `n` valeurs d'un tuple, écris `formula(*values)` : l'étoile déplie le tuple en `n` arguments.
- `implies(p, q)` : la valeur de vérité de $p \Rightarrow q$, en une ligne, avec `not` et `or` (fiche §2).

La vérification calcule, avec tes deux fonctions, la colonne de « (p et q) ⇒ r », le nombre de lignes d'une table à quatre variables, puis le nom de l'erreur levée pour `n = 0` (avec `raised`, de la cellule des outils).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R4")  # reload your saved file

In [ ]:
with checked("R4.17a"):
    rows = mymath.logic.truth_table(lambda p, q, r: mymath.logic.implies(p and q, r), 3)
    wbm.check("R4.17a", [result for values, result in rows], computed=True)
with checked("R4.17b"):
    wbm.check("R4.17b", len(mymath.logic.truth_table(lambda a, b, c, d: a or d, 4)), computed=True)
with checked("R4.17c"):
    wbm.check("R4.17c", raised(mymath.logic.truth_table, lambda: True, 0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r4_logic.py", "test_truth_table_ or test_implies_")

### Ex R4.19 — Chercher un contre-exemple de 0 à N · `find_counterexample` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer la recherche systématique d'un contre-exemple, et comprendre ce qu'elle prouve et ne prouve pas.  
**Prérequis :** fiche §6 ; R4.17 · **mymath :** `logic.py` · **Parcours :** C, E

Écris `find_counterexample(predicate, n_max)` dans `mon_travail/mymath/logic.py` : essaie les entiers $n = 0, 1, \dots, n_{\max}$ **dans l'ordre**, et rends le premier pour lequel `predicate(n)` est faux ; si aucun ne l'est, rends `None`. `predicate` représente la phrase « pour tout $n$, $P(n)$ ».

La vérification cherche :
- a) le premier $n$ pour lequel « $n! < 10^6$ » est faux ;
- b) si « $n^3 - n$ est divisible par 3 » tient de 0 à 1 000 (ta fonction doit alors rendre `None`) ;
- c) un contre-exemple de « $n > 0$ » (attention à $n = 0$) ;
- d) un contre-exemple de « $n < 20$ » avec $n_{\max} = 20$ (attention à $n_{\max}$).

Puis, dans ta copie : la réponse de b) démontre-t-elle que $n^3 - n$ est toujours divisible par 3 ? Démontre-le (écris $n^3 - n = (n - 1)\,n\,(n + 1)$).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R4")  # reload your saved file

In [ ]:
with checked("R4.19a"):
    wbm.check("R4.19a", mymath.logic.find_counterexample(lambda n: math.factorial(n) < 10**6, 50), computed=True)
with checked("R4.19b"):
    wbm.check("R4.19b", mymath.logic.find_counterexample(lambda n: (n**3 - n) % 3 == 0, 1000) is None,
              computed=True)
with checked("R4.19c"):
    wbm.check("R4.19c", mymath.logic.find_counterexample(lambda n: n > 0, 10) == 0, computed=True)
with checked("R4.19d"):
    wbm.check("R4.19d", mymath.logic.find_counterexample(lambda n: n < 20, 20) == 20, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r4_logic.py", "test_find_counterexample_")

### Ex R4.21 — Prédire : n² + n + 41 est-il toujours un nombre premier ? 🔮 ★ ⏱️ 15 min
**Objectif :** voir une propriété vraie sur des dizaines de cas tomber, et comprendre pourquoi des exemples ne démontrent rien.  
**Prérequis :** fiche §6 ; R4.19 · **Parcours :** C, E

Calcule de tête $n^2 + n + 41$ pour $n = 0, 1, 2, 3$ : tu obtiens 41, 43, 47, 53, tous des nombres premiers. **Avant d'exécuter quoi que ce soit**, prédis :
- a) $n^2 + n + 41$ est-il premier pour **tout** entier $n \ge 0$ ? (`True` ou `False`)
- b) le plus petit entier $n$ pour lequel il ne l'est pas, ou `-1` si tu penses qu'il n'y en a pas.

Écris tes deux prédictions, vérifie-les, puis lance l'expérience (elle se sert de `is_prime`, de la cellule des outils ; tu pourrais aussi utiliser ta fonction `find_counterexample`).

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R4_21a = ...  # a) True or False, written BEFORE running the experiment below
prediction_R4_21b = ...  # b) an integer (-1 if you think there is none)

In [ ]:
for letter, prediction in zip("ab", [prediction_R4_21a, prediction_R4_21b]):
    wbm.check(f"R4.21{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes deux prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R4_21a, prediction_R4_21b]):
    print("⏳ Écris d'abord tes deux prédictions, puis exécute à nouveau.")
else:
    values = [(n, n * n + n + 41) for n in range(46)]
    for n, value in values[:5] + values[37:44]:
        print(f"n = {n:2d} : n² + n + 41 = {value:5d}   premier : {is_prime(value)}")
    first = next(n for n, value in values if not is_prime(value))
    print("premier n pour lequel ce n'est pas un nombre premier :", first, "->", first * first + first + 41)

**Après l'expérience.** c) Combien d'entiers $n$ de 0 à 39 donnent un nombre premier ? Puis, dans ta copie : explique **sans calculer** pourquoi $n = 41$ ne peut pas donner un nombre premier.

In [ ]:
answer_R4_21c = ...  # c) an integer

In [ ]:
wbm.check("R4.21c", answer_R4_21c)

## Partie B · Les ensembles

Fiche §4. Les ensembles de Python, puis les lois de De Morgan en image.

### Ex R4.20 — Les ensembles de Python : `set`, `|`, `&`, `-` 📦 ★ ⏱️ 10 min
**Objectif :** calculer unions, intersections, différences et inclusions avec les ensembles de Python.  
**Prérequis :** fiche §4 ; workbookIA 0A §100.3.4 (ensembles) · **Parcours :** C, E

`set("mot")` est l'ensemble des **lettres** d'un mot, sans répétition ni ordre. On pose `A = set("mathematiques")` et `B = set("statistiques")`. Avec les opérateurs de Python (`|` union, `&` intersection, `-` différence, `<=` inclusion, `len` cardinal, `==` égalité), calcule en une ligne chacun :
- a) le nombre de lettres différentes du mot « anticonstitutionnellement » ;
- b) les lettres communes à A et B (un ensemble) ;
- c) le nombre d'éléments de $A \cup B$ ;
- d) $B \subset A$ ? (`True` ou `False`)
- e) les lettres de A qui ne sont pas dans B (un ensemble) ;
- f) les mots « algorithme » et « logarithme » ont-ils le même ensemble de lettres ?

Puis, dans ta copie : explique la réponse de c) à partir de celle de d).

In [ ]:
A = set("mathematiques")
B = set("statistiques")

answer_R4_20a = ...  # a) an integer
answer_R4_20b = ...  # b) a set
answer_R4_20c = ...  # c) an integer
answer_R4_20d = ...  # d) True or False
answer_R4_20e = ...  # e) a set
answer_R4_20f = ...  # f) True or False

In [ ]:
for letter, answer in zip("abcdef", [answer_R4_20a, answer_R4_20b, answer_R4_20c, answer_R4_20d,
                                       answer_R4_20e, answer_R4_20f]):
    wbm.check(f"R4.20{letter}", answer)

### Ex R4.23 — Diagrammes de Venn 🎨 ★ ⏱️ 15 min
**Objectif :** voir les opérations sur les ensembles, et vérifier les lois de De Morgan sur une figure.  
**Prérequis :** fiche §4 ; R0 (matplotlib) · **Parcours :** C

La cellule ci-dessous fabrique l'univers $E$ (une grille de points) et deux disques $A$ et $B$ : `in_a` et `in_b` sont des tableaux de booléens qui disent, pour chaque point, s'il est dans $A$ ou dans $B$. Sur ces tableaux, NumPy calcule point par point : `~` est « non », `&` est « et », `|` est « ou » (mets des parenthèses : `~in_a & ~in_b`).

Complète `venn_regions(in_a, in_b)`, qui rend la liste des quatre tableaux :
1. $\bar{A} \cap \bar{B}$ (ni dans $A$, ni dans $B$) ;
2. $\overline{A \cup B}$ ;
3. $\bar{A} \cup \bar{B}$ ;
4. $\overline{A \cap B}$.

La vérification les dessine, puis regarde quelques points. Dans ta copie : quelles régions coïncident, et quelle loi de la fiche cela illustre-t-il ?

In [ ]:
def show_regions(xs, ys, regions, titles):
    """Draw four boolean masks of the grid (xs, ys) as shaded Venn diagrams, in a 2 × 2 figure."""
    fig, axes = plt.subplots(2, 2, figsize=(9, 5.4))
    for ax, mask, title in zip(axes.ravel(), regions, titles):
        ax.contourf(xs, ys, np.asarray(mask, dtype=float), levels=[0.5, 1.5], colors=["C0"], alpha=0.45)
        for centre, name in ((-0.55, "A"), (0.55, "B")):
            ax.add_patch(plt.Circle((centre, 0), 1, fill=False, linewidth=1.6))
            ax.text(centre * 2.3, 0.8, name, fontsize=12, ha="center", fontweight="bold")
        ax.add_patch(plt.Rectangle((-2.2, -1.5), 4.4, 3, fill=False, linewidth=1.4))
        ax.set_title(title, fontsize=11)
        ax.set_xlim(-2.3, 2.3)
        ax.set_ylim(-1.6, 1.6)
        ax.set_aspect("equal")
        ax.axis("off")
    fig.tight_layout()
    plt.show()


# the universe E: a grid of points; A and B: two discs
xs, ys = np.meshgrid(np.linspace(-2.2, 2.2, 300), np.linspace(-1.5, 1.5, 200))
in_a = (xs + 0.55) ** 2 + ys**2 <= 1
in_b = (xs - 0.55) ** 2 + ys**2 <= 1
TITLES = ["non A et non B", "non (A ou B)", "non A ou non B", "non (A et B)"]

In [ ]:
def venn_regions(in_a, in_b):
    """[not A and not B, not (A or B), not A or not B, not (A and B)]: four boolean arrays, point by point."""
    raise NotImplementedError("combine in_a and in_b with ~, & and |")

In [ ]:
with checked("R4.23"):
    show_regions(xs, ys, venn_regions(in_a, in_b), TITLES)

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R4.23 vérifications"):
    regions = [np.asarray(mask, dtype=bool) for mask in venn_regions(in_a, in_b)]
    outside, only_a, both = (0, 140), (100, 60), (100, 150)      # three points of the grid (row, column)
    problems = []
    if len(regions) != 4 or any(mask.shape != in_a.shape for mask in regions):
        problems.append("quatre tableaux de booléens de la forme de in_a")
    else:
        if not regions[0][outside] or regions[0][only_a]:
            problems.append("le 1ᵉʳ : les points qui ne sont ni dans A ni dans B")
        if not regions[2][only_a] or regions[2][both]:
            problems.append("le 3ᵉ : les points qui manquent à A ou à B")
        if not np.array_equal(regions[0], regions[1]) or not np.array_equal(regions[2], regions[3]):
            problems.append("le 2ᵉ égal au 1ᵉʳ, et le 4ᵉ égal au 3ᵉ, point par point")
    if problems:
        print("❌ Ex R4.23 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R4.23 : les deux paires de régions coïncident point par point ; écris dans ta copie la loi "
              "qu'elles illustrent.")

## Partie C · Σ, Π et récurrence

Fiche §5, §7 et §8. Σ et Π dans `mymath/logic.py`, une formule vérifiée mille fois puis démontrée, deux récurrences fausses, et une démonstration mise au propre.

### Ex R4.18 — Σ et Π en Python · `sum_over`, `prod_over` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer Σ et Π avec leurs bornes incluses, et les cas de la somme et du produit vides.  
**Prérequis :** fiche §5 · **mymath :** `logic.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en R2 et R3). Ouvre ta copie `mon_travail/mymath/logic.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Les tests comparent tes résultats à SymPy et à une évaluation directe.

Écris `sum_over(f, start, stop)` et `prod_over(f, start, stop)` : la somme et le produit des `f(k)` pour `k` allant de `start` à `stop`, **les deux bornes incluses**, comme $\sum_{k=\text{start}}^{\text{stop}} f(k)$ et $\prod_{k=\text{start}}^{\text{stop}} f(k)$. Une boucle `for` sur un `range` et un total suffisent ; réfléchis à la valeur de départ du total, et à ce que rendent les deux fonctions quand `stop < start` (fiche §5).

La vérification calcule $\sum_{k=1}^{5} k(k+1)$, une somme vide, $\prod_{k=1}^{10} 2$ et un produit vide, puis $\sum_{k=1}^{10} \frac{1}{2^k}$.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R4")  # reload your saved file

In [ ]:
with checked("R4.18a"):
    wbm.check("R4.18a", [mymath.logic.sum_over(lambda k: k * (k + 1), 1, 5), mymath.logic.sum_over(lambda k: k, 5, 4),
                         mymath.logic.prod_over(lambda k: 2, 1, 10), mymath.logic.prod_over(lambda k: k, 3, 2)],
              computed=True)
with checked("R4.18b"):
    wbm.check("R4.18b", mymath.logic.sum_over(lambda k: 1 / 2**k, 1, 10), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r4_logic.py", "test_sum_over_ or test_prod_over_")

### Ex R4.22 — Vérifier une formule de somme de 1 à 1 000, puis la démontrer 🔬 ★★ ⏱️ 20 min
**Objectif :** vérifier une formule sur mille cas, voir une formule fausse qui colle aux premiers cas, puis démontrer la vraie.  
**Prérequis :** fiche §5 et §7 ; R4.12 · **Parcours :** C

On conjecture la **formule A** : pour tout entier $n \ge 1$, $\displaystyle\sum_{k=1}^{n} k^3 = \left(\frac{n(n+1)}{2}\right)^2$. Un camarade propose la **formule B** : $\displaystyle\sum_{k=1}^{n} k^3 = \frac{n^2(n+1)^2}{4} + (n - 1)(n - 2)(n - 3)$.

Complète les trois fonctions (avec `//` pour les divisions, qui tombent juste : les résultats restent des entiers exacts). La vérification calcule :
- a) $\sum_{k=1}^{1000} k^3$ ;
- b) le nombre d'entiers $n$ de 1 à 1 000 pour lesquels la formule A **diffère** de la somme ;
- c) le plus petit $n \ge 1$ pour lequel la formule B diffère de la somme.

Puis l'expérience trace les deux courbes. Dans ta copie : démontre la formule A par récurrence (comme R4.12 : isole le dernier terme, puis factorise par $(n + 1)^2$), et explique pourquoi la formule B est juste pour $n = 1, 2, 3$.

In [ ]:
def sum_of_cubes(n):
    """1³ + 2³ + … + n³, added term by term (a loop, or sum over a range)."""
    raise NotImplementedError("add the cubes from 1 to n")


def formula_a(n):
    """(n(n + 1)/2)², computed with // so that it stays an exact integer."""
    raise NotImplementedError("write formula A")


def formula_b(n):
    """The classmate's formula: n²(n + 1)²/4 + (n − 1)(n − 2)(n − 3), with //."""
    raise NotImplementedError("write formula B")

In [ ]:
with checked("R4.22a"):
    wbm.check("R4.22a", sum_of_cubes(1000), computed=True)
with checked("R4.22b"):
    wbm.check("R4.22b", sum(sum_of_cubes(n) != formula_a(n) for n in range(1, 1001)), computed=True)
with checked("R4.22c"):
    wbm.check("R4.22c", next(n for n in range(1, 1001) if sum_of_cubes(n) != formula_b(n)), computed=True)

**L'expérience** : la somme et la formule B pour $n = 1$ à 12.

In [ ]:
with checked("R4.22 figure"):
    ns = list(range(1, 13))
    cubes, proposed = [sum_of_cubes(n) for n in ns], [formula_b(n) for n in ns]   # before the figure
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(ns, cubes, "o-", label="1³ + 2³ + … + n³")
    ax.plot(ns, proposed, "s--", label="formule B")
    ax.set_yscale("log")
    ax.set_xlabel("n")
    ax.set_ylabel("valeur (échelle logarithmique)")
    ax.set_title("La formule B colle au début, puis décroche")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

### Ex R4.24 — Chasse à l'erreur : une « démonstration » par récurrence qui prouve le faux 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer ce qui manque ou ce qui ne marche pas dans une récurrence.  
**Prérequis :** fiche §7 et §8 ; R4.12 · **Parcours :** C, E

Deux « démonstrations » par récurrence prouvent des choses fausses. **Sans rien exécuter d'abord**, trouve ce qui ne va pas.

**Preuve 1.** « Pour tout entier $n \ge 0$, $5^n + 1$ est divisible par 4. *Hérédité* : soit $n \ge 0$ tel que $5^n + 1 = 4m$, avec $m$ entier. Alors $5^{n+1} + 1 = 5 \times 5^n + 1 = 5(5^n + 1) - 4 = 20m - 4 = 4(5m - 1)$, divisible par 4. Donc, par récurrence, la propriété est vraie pour tout $n \ge 0$. »

**Preuve 2.** « Dans tout groupe de $n \ge 1$ boules, toutes les boules ont la même couleur. »

| | Ligne |
|---|---|
| A | Pour $n \ge 1$, notons $P(n)$ : « dans tout groupe de $n$ boules, toutes ont la même couleur ». |
| B | Initialisation : un groupe d'une seule boule n'a qu'une couleur, donc $P(1)$ est vraie. |
| C | Hérédité : soit $n \ge 1$ tel que $P(n)$ est vraie, et un groupe de $n + 1$ boules numérotées de 1 à $n + 1$. |
| D | Les boules 1 à $n$ forment un groupe de $n$ boules : d'après $P(n)$, elles ont toutes la même couleur. |
| E | Les boules 2 à $n + 1$ forment aussi un groupe de $n$ boules : elles ont toutes la même couleur. |
| F | La boule 2 appartient aux deux groupes, donc les deux groupes ont la même couleur, et les $n + 1$ boules aussi. |
| G | Par récurrence, $P(n)$ est vraie pour tout $n \ge 1$. |

- a) Preuve 1 : quelle étape manque ? 1 pour l'initialisation, 2 pour l'hérédité.
- b) Preuve 1 : combien d'entiers $n$ de 0 à 20 rendent $5^n + 1$ divisible par 4 ?
- c) Preuve 2 : la lettre de la ligne fausse (un texte).
- d) Preuve 2 : la plus petite valeur de $n$ pour laquelle le texte ne démontre pas « $P(n) \Rightarrow P(n + 1)$ ».
- e) Preuve 2 : $P(2)$ est-elle vraie ?

Puis, dans ta copie : pourquoi la preuve 1 ne peut-elle pas être réparée, alors que son hérédité est juste ? Et que montre la preuve 2 sur l'hérédité ?

In [ ]:
answer_R4_24a = ...  # a) 1 or 2
answer_R4_24b = ...  # b) an integer
answer_R4_24c = ...  # c) a letter, as text
answer_R4_24d = ...  # d) an integer
answer_R4_24e = ...  # e) True or False

In [ ]:
for letter, answer in zip("abcde", [answer_R4_24a, answer_R4_24b, answer_R4_24c, answer_R4_24d, answer_R4_24e]):
    wbm.check(f"R4.24{letter}", answer)

### Ex R4.25 — Rédiger une démonstration propre en LaTeX 🛠️ ★ ⏱️ 20 min
**Objectif :** mettre au propre une démonstration, avec ses formules en LaTeX, comme dans un rapport ou un notebook partagé.  
**Prérequis :** fiche §8 ; R4.12 ; R0 (LaTeX) · **Parcours :** C

Recopie **au propre**, dans la cellule Markdown ci-dessous, ta démonstration par récurrence de R4.12 ($\sum_{k=1}^{n} k = \frac{n(n+1)}{2}$), en LaTeX. Elle doit contenir :
- l'énoncé de la propriété, avec son quantificateur, et la définition de $P(n)$ ;
- trois paragraphes titrés *Initialisation*, *Hérédité*, *Conclusion* ;
- dans l'hérédité, un calcul centré entre `$$`, qui fait apparaître la somme jusqu'à $n$ et l'endroit où l'hypothèse de récurrence sert ;
- un signe de fin, `$\blacksquare$`.

Commandes utiles : `\sum_{k=1}^{n} k`, `\frac{n(n+1)}{2}`, `\ge`, `\Rightarrow`, `\displaystyle`. Exécute la cellule pour voir le rendu, corrige ce qui s'affiche mal, puis compare avec le modèle de `05_solutions.ipynb`.

✍️ **Ma démonstration** (double-clique sur cette cellule pour l'écrire, puis Maj + Entrée pour l'afficher)

**Propriété.** Pour tout entier $n \ge 1$, …

*Démonstration.* …

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Dresses-tu une table de vérité, et nies-tu une phrase avec « et », « ou », « pour tout », « il existe » ?
2. Distingues-tu implication, réciproque et contraposée, et calcules-tu unions, intersections, cardinaux, sommes Σ et produits Π ?
3. Choisis-tu un raisonnement adapté, et sais-tu rédiger une récurrence complète ?

**La suite** : R5, *Suites*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.